In [ ]:
import numpy as np
import pandas as pd

from uncertaintyx.fit.eiv.numpy import EIV
from uncertaintyx.fit.randomsampling import Bootstrap
from uncertaintyx.fit.regression import HomoscedasticRegression
from uncertaintyx.m.numpy import ToM
from uncertaintyx.plot.plots import MatrixPlot
from uncertaintyx.plot.plots import RegressionPlot

In [ ]:
class M2(ToM):
    """
    Empirical model function to fit data in Lee et al. (2010, Figure 2,
    https://www.ioccg.org/groups/Software_OCA/QAA_v5.pdf).
    """

    def __init__(self):
        """The model function."""

        def f(p, x):
            a, c = p
            return a * (1.0 - 1.2 * np.exp(-c * x))

        super().__init__(f)

    def jac_p(self, par: np.ndarray, x: np.ndarray) -> np.ndarray:
        a, c = par
        term = np.exp(-c * x)
        return np.stack([1.0 - 1.2 * term, a * 1.2 * x * term], axis=-1)

    def jac_x(self, par: np.ndarray, x: np.ndarray) -> np.ndarray:
        a, c = par
        term = np.exp(-c * x)
        return a * 1.2 * c * term

    def estimate(
        self,
        x: np.ndarray | None = None,
        y: np.ndarray | None = None,
        preset: str | None = None,
    ) -> np.ndarray:
        return np.array([2.0, 0.9])

In [ ]:
class M3(ToM):
    """
    Empirical model function to fit data in Lee et al. (2010, Figure 3,
    https://www.ioccg.org/groups/Software_OCA/QAA_v5.pdf).
    """

    def __init__(self):
        def f(p, x):
            """The model function."""
            a, b, c = p
            return a + b / (c + x)

        super().__init__(f)

    def jac_p(self, par: np.ndarray, x: np.ndarray) -> np.ndarray:
        _, b, c = par
        term = 1.0 / (c + x)
        return np.stack([np.ones_like(term), term, -b * term * term], axis=-1)

    def jac_x(self, par: np.ndarray, x: np.ndarray) -> np.ndarray:
        _, b, c = par
        term = 1.0 / (c + x)
        return -b * term * term

    def estimate(
        self,
        x: np.ndarray | None = None,
        y: np.ndarray | None = None,
        preset: str | None = None,
    ) -> np.ndarray:
        return np.array([0.015, 0.002, 0.6])

# Figure 2

In [ ]:
df = pd.read_csv("../test/resources/fig2.csv", sep=";")
x = df["X"].values.reshape((-1, 1))
y = df["Y"].values.reshape((-1, 1))

In [ ]:
result = Bootstrap(HomoscedasticRegression(EIV())).fit(M2(), x, y)

In [ ]:
print("fit parameters = ", result.popt)
print("fit parameters uncertainty = ", result.punc)
print("fit parameters covariance matrix = ", result.pcov)
print("fit residual variance = ", result.yvar_r)

In [ ]:
RegressionPlot(result).plot(
    x,
    y,
    title="Replica of Lee et al. (2010, Figure 2)",
    xlabel=r"$r(443~\mathrm{nm})$ / $r(555~\mathrm{nm})$",
    ylabel=r"$\eta$",
    xrange=(0.5, 4.5),
    yrange=(-0.5, 2.5),
)

In [ ]:
MatrixPlot().plot(
    result.ycov_p(np.linspace(0.5, 4.5, 1000)),
    xlabel=r"$r(443~\mathrm{nm})$ / $r(555~\mathrm{nm})$",
    ylabel=r"$r(443~\mathrm{nm})$ / $r(555~\mathrm{nm})$",
    xrange=(0.5, 4.5),
    yrange=(0.5, 4.5),
    cmap="cividis",
    cbar_label=r"variance-covariance $U_p(\eta)$",
    cbar_max=0.035,
    cbar_min=-0.035,
)

# Figure 3

In [ ]:
df = pd.read_csv("../test/resources/fig3.csv", sep=";")
x = df["X"].values.reshape((-1, 1))
y = df["Y"].values.reshape((-1, 1))

In [ ]:
result = Bootstrap(HomoscedasticRegression(EIV())).fit(M3(), x, y)

In [ ]:
print("fit parameters = ", result.popt)
print("fit parameters uncertainty = ", result.punc)
print("fit parameters covariance matrix = ", result.pcov)
print("fit residual variance = ", result.yvar_r)

In [ ]:
RegressionPlot(result).plot(
    x,
    y,
    title="Replica of Lee et al. (2010, Figure 3)",
    xlabel=r"$r(443~\mathrm{nm})$ / $r(555~\mathrm{nm})$",
    ylabel=r"$S$ ($\mathrm{nm}^{-1}$)",
    xrange=(0.050, 9.950),
    yrange=(0.005, 0.035),
)

In [ ]:
MatrixPlot().plot(
    result.ycov_p(np.linspace(0.05, 9.95, 1000)),
    xlabel=r"$r(443~\mathrm{nm})$ / $r(555~\mathrm{nm})$",
    ylabel=r"$r(443~\mathrm{nm})$ / $r(555~\mathrm{nm})$",
    xrange=(0.05, 9.95),
    yrange=(0.05, 9.95),
    cmap="viridis",
    cbar_label=r"variance-covariance $U_p(S)$ ($\mathrm{nm}^{-2}$)",
    cbar_max=3.0e-05,
    cbar_min=0.0e-05,
)